# SuperTrend

SuperTrend Trend-Following \
Pure trend strategy with built-in ATR trailing stop. \
It uses ATR(14) for dynamic stops/tolerance (adapts to volatility). \
Extremely popular in crypto perpetuals for all timeframes; maximizes profit by riding trends while cutting losses fast.

__How SuperTrend Algorithm Determines Entry/Exit:__
- Combines ATR volatility bands with trend direction.
- Long Entry: SuperTrend flips from red to green (price closes above upper band).
- Short Entry: SuperTrend flips from green to red (price closes below lower band).
- Exit: SuperTrend flips opposite OR the built-in ATR trailing stop is hit (the SuperTrend line itself acts as dynamic stop).
- Extremely clean, low-lag, and maximizes trend capture while protecting capital.

## Configuration: automatic

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import StrategyConfig
from engine.visualization import build_chart

In [ ]:
# Automatic config: canonical handles from the three configurators (project-wide defaults).
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
import dataclasses
from engine.data_configurator import ACTIVE, load_data, save_result
from engine.strategy_configurator import StrategyConfig, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE

DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = StrategyConfig()    # engine/strategy_configurator.py (signal knobs)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

## Configuration: manual

Per-notebook overrides on top of the automatic config above. Each cell applies
`dataclasses.replace` to one handle; **leave a dict empty (or `EXIT_POLICY = None`)
to keep that dimension automatic**. Everything below this chapter uses only
`df`, `SYMBOL`, `INTERVAL`, `STRATEGY_CONFIG`, `EXIT_POLICY`, `TRADING_CONFIG`.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic StrategyConfig().
STRATEGY_OVERRIDES = {}      # e.g. {"supertrend_period": 7, "supertrend_mult": 2.5, "adx_threshold": 20}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = each strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

In [ ]:
# Resolve the final inputs the rest of the notebook uses. (Runs after overrides.)
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

## SuperTrend

In [ ]:
# Import SuperTrend strategy
from engine.strategies import SuperTrendStrategy

In [ ]:
# Backtest SuperTrend strategy
strategy = SuperTrendStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [ ]:
# Dollar P&L
INITIAL_BALANCE = 100  # USD

balance = INITIAL_BALANCE
peak = balance
max_dd = 0

print(f"\n{'─' * 45}")
print(f"  Dollar P&L (starting ${INITIAL_BALANCE:,.2f})")
print(f"{'─' * 45}")

for i, t in enumerate(result.trades, 1):
    prev = balance
    balance *= (1 + t.pnl_bps / 10_000)
    peak = max(peak, balance)
    max_dd = max(max_dd, (peak - balance) / peak)
    pnl = balance - prev
    print(f"  #{i:3d}  {t.direction.value:5s}  {pnl:+8.2f}  →  ${balance:,.2f}")

print(f"{'─' * 45}")
print(f"  Final balance  : ${balance:,.2f}")
print(f"  Net profit     : ${balance - INITIAL_BALANCE:+,.2f}")
print(f"  Return         : {(balance / INITIAL_BALANCE - 1):+.2%}")
print(f"  Max drawdown   : {max_dd:.2%}")

In [ ]:
# SuperTrend strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Inverse SuperTrend

In [ ]:
# Import inverse SuperTrend strategy
from engine.strategies import InverseSuperTrendStrategy

In [ ]:
# Backtest inverse SuperTrend strategy
strategy = InverseSuperTrendStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [ ]:
# Dollar P&L
INITIAL_BALANCE = 100  # USD

balance = INITIAL_BALANCE
peak = balance
max_dd = 0

print(f"\n{'─' * 45}")
print(f"  Dollar P&L (starting ${INITIAL_BALANCE:,.2f})")
print(f"{'─' * 45}")

for i, t in enumerate(result.trades, 1):
    prev = balance
    balance *= (1 + t.pnl_bps / 10_000)
    peak = max(peak, balance)
    max_dd = max(max_dd, (peak - balance) / peak)
    pnl = balance - prev
    print(f"  #{i:3d}  {t.direction.value:5s}  {pnl:+8.2f}  →  ${balance:,.2f}")

print(f"{'─' * 45}")
print(f"  Final balance  : ${balance:,.2f}")
print(f"  Net profit     : ${balance - INITIAL_BALANCE:+,.2f}")
print(f"  Return         : {(balance / INITIAL_BALANCE - 1):+.2%}")
print(f"  Max drawdown   : {max_dd:.2%}")

In [ ]:
# inverse SuperTrend strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Adaptive SuperTrend

On each bar, ADX is checked. \
If ADX ≥ 25 (trending), a SuperTrend flip triggers a trade in the same direction (trend-following, normal SuperTrend). \
If ADX < 25 (ranging), the same flip triggers the opposite direction (mean-reversion, inverse SuperTrend). \
Exits also respect the current regime — so if the market shifts from trending to ranging mid-trade, the exit condition matches what made sense at entry.

The `adx_threshold` parameter is now a `StrategyConfig` knob — tune it from the manual chapter (`STRATEGY_OVERRIDES = {"adx_threshold": 20}`). \
It controls the regime switch:
- lower (20) makes it more aggressive about calling "trending"
- higher (30) makes it pickier

In [ ]:
# Import adaptive SuperTrend strategy
from engine.strategies.supertrend_adaptive import AdaptiveSuperTrendStrategy

In [ ]:
# # Backtest adaptive SuperTrend strategy
strategy = AdaptiveSuperTrendStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [ ]:
# Dollar P&L
INITIAL_BALANCE = 100  # USD

balance = INITIAL_BALANCE
peak = balance
max_dd = 0

print(f"\n{'─' * 45}")
print(f"  Dollar P&L (starting ${INITIAL_BALANCE:,.2f})")
print(f"{'─' * 45}")

for i, t in enumerate(result.trades, 1):
    prev = balance
    balance *= (1 + t.pnl_bps / 10_000)
    peak = max(peak, balance)
    max_dd = max(max_dd, (peak - balance) / peak)
    pnl = balance - prev
    print(f"  #{i:3d}  {t.direction.value:5s}  {pnl:+8.2f}  →  ${balance:,.2f}")

print(f"{'─' * 45}")
print(f"  Final balance  : ${balance:,.2f}")
print(f"  Net profit     : ${balance - INITIAL_BALANCE:+,.2f}")
print(f"  Return         : {(balance / INITIAL_BALANCE - 1):+.2%}")
print(f"  Max drawdown   : {max_dd:.2%}")

In [ ]:
# adaptive SuperTrend strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Live signals

Live mode:
- It generates signals, it does not place orders. \
There's no exchange API key, no order execution. It tells you when to enter/exit.
- State persists \
If you stop and restart, it remembers whether you're in a position via its SQLite state file under data/live/.
- Circuit breaker \
If Bybit is unreachable 10 times in a row, it stops automatically instead of spinning forever.
- Chart updates in place \
Automatic: the chart refreshes in the browser every poll_seconds.

To actually execute trades automatically, you need to add authenticated Bybit order placement on top of the signal output.

If you don't want to re-download automatically, two edits locally:
1. visualization.py — add auto_refresh: int = 0 parameter to build_chart, and after fig.write_html(save_path):
pythonif auto_refresh > 0:
    with open(save_path, "r") as f:
        html = f.read()
    meta_tag = f'<meta http-equiv="refresh" content="{auto_refresh}">'
    html = html.replace("<head>", f"<head>{meta_tag}", 1)
    with open(save_path, "w") as f:
        f.write(html)
2. live.py — add auto_refresh=self.poll_seconds to the build_chart() call in _tick().

### From CLI (command line interface)

- runs in a loop
- polls (refreshes) Bybit every 30 seconds
- persists state to SQLite (survives restarts)
- writes a chart under data/live/ each tick
- handles SIGTERM/Ctrl+C gracefully

In [ ]:
python -m engine \
    --strategy supertrend_adaptive \
    --mode live \
    --symbol BTCUSDT \
    --interval 15 \
    --candles 500 \
    --poll 30

### From a notebook cell

In [ ]:
from engine.live import LiveEngine
from engine.data_configurator import LIVE_DIR
from engine.strategy_configurator import StrategyConfig
from engine.strategies.supertrend_inv import InverseSuperTrendStrategy

SYMBOL   = "BTCUSDT"
INTERVAL = "15"

config = StrategyConfig()
strategy = InverseSuperTrendStrategy(config)

engine = LiveEngine(
    strategy=strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=30,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{strategy.name}.db"),
)

engine.run()  # blocks until Ctrl+C or kernel interrupt

## Live-mode for several strategies simultaneously

### From CLI

- one terminal per strategy
- no need to set --db/--save: the defaults already write per-strategy paths under data/live/ (data/live/<strategy>.db and data/live/<symbol>_<interval>_<strategy>.html), so concurrent runs don't clobber each other

In [ ]:
python -m engine --strategy supertrend          --mode live --interval 15 &
python -m engine --strategy supertrend_inv      --mode live --interval 15 &
python -m engine --strategy supertrend_adaptive --mode live --interval 15 &

### From a notebook cell

In [ ]:
import threading
from engine.live import LiveEngine
from engine.data_configurator import LIVE_DIR
from engine.strategy_configurator import StrategyConfig
from engine.strategies import SuperTrendStrategy
from engine.strategies.supertrend_inv import InverseSuperTrendStrategy
from engine.strategies.supertrend_adaptive import AdaptiveSuperTrendStrategy

SYMBOL   = "BTCUSDT"
INTERVAL = "15"
config   = StrategyConfig()

strategies = [
    SuperTrendStrategy(config),
    InverseSuperTrendStrategy(config),
    AdaptiveSuperTrendStrategy(config),
]

threads = []
for strat in strategies:
    engine = LiveEngine(
        strategy=strat,
        symbol=SYMBOL,
        interval=INTERVAL,
        num_candles=500,
        poll_seconds=30,
        chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{strat.name}.html"),
        db_path=str(LIVE_DIR / f"{strat.name}.db"),   # separate DB per strategy
    )
    t = threading.Thread(target=engine.run, name=strat.name, daemon=True)
    threads.append(t)
    t.start()
    print(f"Started: {strat.name}")

# Blocks until you interrupt the kernel
for t in threads:
    t.join()